# 06 — Dixon-Coles Features

Dixon-Coles (1997) is the gold standard statistical model for football score prediction.  
Instead of predicting win/draw/loss directly, it models **how many goals each team scores** using Poisson distributions:

```
home_goals ~ Poisson(attack_home × defense_away × home_advantage)
away_goals ~ Poisson(attack_away × defense_home)
```

Each team gets two parameters:
- **attack**: how many goals they tend to score (relative to league average)
- **defense**: how many goals they tend to concede (relative to league average)

From the full score distribution we derive win/draw/loss probabilities.  
These 3 probabilities become **new features** in our ensemble — capturing goal-level dominance that ELO alone misses.

**Key design choice: Time-weighting**  
Recent matches matter more than old ones. We apply exponential decay:  
`weight = exp(-ξ × days_since_match)` where ξ = 0.003  
A match from 3 years ago weighs ~1/3 of a match from last week.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, json, joblib
warnings.filterwarnings('ignore')

from pathlib import Path
from scipy.optimize import minimize
from scipy.stats import poisson
from sklearn.metrics import accuracy_score, f1_score, log_loss
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.ensemble import VotingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
import xgboost as xgb

sns.set_theme(style='darkgrid')
PROCESSED_DIR = Path('../data/processed')
MODELS_DIR    = Path('../models')

print('Libraries loaded.')

Libraries loaded.


## Step 1: Load Match Data & Fit Dixon-Coles

We fit on ALL competitive matches (excluding friendlies) — time-weighted so recent form drives the ratings.

In [2]:
matches = pd.read_csv(PROCESSED_DIR / 'matches_clean.csv', parse_dates=['date'])

# Use all competitive matches for DC fitting (friendlies excluded)
# EXP-27 insight: quality-only filtering hurts log loss (-0.023) because sparse data
# DC is used for goal distribution, not global ranking -- ELO handles cross-confederation ranking
comp = matches[matches['tournament_category'] != 'friendly'].copy()
comp = comp.dropna(subset=['home_score', 'away_score'])
comp['home_score'] = comp['home_score'].astype(int)
comp['away_score'] = comp['away_score'].astype(int)

print(f'Competitive matches for DC fitting: {len(comp):,}')
print(f'Date range: {comp["date"].min().date()} to {comp["date"].max().date()}')
print(f'Unique teams: {len(set(comp["home_team"]) | set(comp["away_team"]))}')


Competitive matches for DC fitting: 38,617
Date range: 1884-01-26 to 2026-03-31
Unique teams: 317


In [3]:
# -- Time-weighting (EXP-17: xi=0.002 instead of 0.003) --
# xi=0.003 ranked Australia #1 (too aggressive, recent weak-opponent wins inflated)
# xi=0.002 gives better balance: 1yr ago = 48%, 3yr ago = 11%, 5yr ago = 2.6%
XI = 0.002
reference_date = comp['date'].max()
comp['days_ago'] = (reference_date - comp['date']).dt.days
comp['weight'] = np.exp(-XI * comp['days_ago'])

print(f'Reference date: {reference_date.date()}')
print(f'[EXP-17] xi={XI}')
print(f'Weight range: {comp["weight"].min():.4f} -> {comp["weight"].max():.4f}')
print(f'Weight 1yr ago: {np.exp(-XI*365):.3f}')
print(f'Weight 3yr ago: {np.exp(-XI*365*3):.3f}')
print(f'Weight 5yr ago: {np.exp(-XI*365*5):.3f}')


Reference date: 2026-03-31
[EXP-17] xi=0.002
Weight range: 0.0000 -> 1.0000
Weight 1yr ago: 0.482
Weight 3yr ago: 0.112
Weight 5yr ago: 0.026


In [4]:
# ── Team index ─────────────────────────────────────────────────────────────────
all_teams = sorted(set(comp['home_team']) | set(comp['away_team']))
team_idx  = {t: i for i, t in enumerate(all_teams)}
n_teams   = len(all_teams)

print(f'Teams to fit: {n_teams}')

# Define arrays here so cell 6 (likelihood) can use them
home_teams_arr = comp['home_team'].values
away_teams_arr = comp['away_team'].values
home_goals_arr = comp['home_score'].values
away_goals_arr = comp['away_score'].values
weights_arr    = comp['weight'].values
print(f'Arrays defined: {len(home_teams_arr):,} matches')

Teams to fit: 317
Arrays defined: 38,617 matches


In [5]:
# Precompute team index arrays once (reused in every likelihood call)
home_idx_arr = np.array([team_idx[t] for t in home_teams_arr])
away_idx_arr = np.array([team_idx[t] for t in away_teams_arr])

# Precompute low-score masks once
_mask_00 = (home_goals_arr == 0) & (away_goals_arr == 0)
_mask_01 = (home_goals_arr == 0) & (away_goals_arr == 1)
_mask_10 = (home_goals_arr == 1) & (away_goals_arr == 0)
_mask_11 = (home_goals_arr == 1) & (away_goals_arr == 1)

def dc_log_likelihood_fast(params):
    attack   = params[:n_teams]
    defense  = params[n_teams:2*n_teams]
    home_adv = params[2*n_teams]
    rho      = params[2*n_teams + 1]
    lam = np.exp(attack[home_idx_arr] + defense[away_idx_arr] + home_adv)
    mu  = np.exp(attack[away_idx_arr] + defense[home_idx_arr])
    tau_vals = np.ones(len(home_goals_arr))
    tau_vals[_mask_00] = 1 - lam[_mask_00] * mu[_mask_00] * rho
    tau_vals[_mask_01] = 1 + lam[_mask_01] * rho
    tau_vals[_mask_10] = 1 + mu[_mask_10]  * rho
    tau_vals[_mask_11] = 1 - rho
    valid = tau_vals > 0
    ll = np.sum(weights_arr[valid] * (
        np.log(tau_vals[valid])
        + poisson.logpmf(home_goals_arr[valid], lam[valid])
        + poisson.logpmf(away_goals_arr[valid], mu[valid])
    ))
    return -ll

def tau(x, y, lam, mu, rho):
    if x == 0 and y == 0: return 1 - lam * mu * rho
    if x == 0 and y == 1: return 1 + lam * rho
    if x == 1 and y == 0: return 1 + mu * rho
    if x == 1 and y == 1: return 1 - rho
    return 1.0

print("Vectorized likelihood defined.")

Vectorized likelihood defined.


## EXP-17 — Tune ξ Decay Rate

Compare ξ=0.001, 0.002, 0.003 on a quick fit to find which puts the right teams at the top.

Expected: Spain, Argentina, France, Brazil in top 5. ξ=0.003 currently ranks Australia #1 (too aggressive).

In [6]:
# EXP-17: Quick xi comparison -- fit DC with 3 different decay rates
from scipy.optimize import minimize as _minimize
from scipy.stats import poisson as _poisson

def quick_dc_fit(xi_val, max_iter=500):
    w = np.exp(-xi_val * comp["days_ago"].values)
    hg = home_goals_arr
    ag = away_goals_arr
    hi = home_idx_arr  # defined in previous cell
    ai = away_idx_arr

    m00 = (hg == 0) & (ag == 0)
    m01 = (hg == 0) & (ag == 1)
    m10 = (hg == 1) & (ag == 0)
    m11 = (hg == 1) & (ag == 1)

    def nll(params):
        atk = params[:n_teams]
        dfn = params[n_teams:2*n_teams]
        ha  = params[2*n_teams]
        rh  = params[2*n_teams + 1]
        lam = np.exp(atk[hi] + dfn[ai] + ha)
        mu  = np.exp(atk[ai] + dfn[hi])
        tau = np.ones(len(hg))
        tau[m00] = 1 - lam[m00]*mu[m00]*rh
        tau[m01] = 1 + lam[m01]*rh
        tau[m10] = 1 + mu[m10]*rh
        tau[m11] = 1 - rh
        valid = tau > 0
        ll = np.sum(w[valid] * (
            np.log(tau[valid])
            + _poisson.logpmf(hg[valid], lam[valid])
            + _poisson.logpmf(ag[valid], mu[valid])
        ))
        return -ll

    eng_i = team_idx.get("England", 0)
    init = np.zeros(2*n_teams + 2)
    init[2*n_teams] = 0.3
    init[2*n_teams+1] = -0.1
    bnds = [(None, None)] * (2*n_teams + 2)
    bnds[eng_i] = (0.0, 0.0)

    res = _minimize(nll, init, method="L-BFGS-B", bounds=bnds,
                    options={"maxiter": max_iter, "ftol": 1e-6})
    atk = res.x[:n_teams]
    dfn = res.x[n_teams:2*n_teams]
    df_r = pd.DataFrame({"team": all_teams, "attack": atk, "defense": dfn})
    df_r["overall"] = df_r["attack"] - df_r["defense"]
    return res.success, df_r.sort_values("overall", ascending=False)

print("EXP-17: Testing xi=0.001, 0.002, 0.003 (~30s each)...")
for xi_val in [0.001, 0.002, 0.003]:
    ok, df_r = quick_dc_fit(xi_val)
    print(f"xi={xi_val}  converged={ok}")
    print(df_r.head(12)[["team","overall"]].to_string(index=False))


EXP-17: Testing xi=0.001, 0.002, 0.003 (~30s each)...
xi=0.001  converged=False
         team  overall
        Japan 2.248878
       Mexico 2.229072
    Australia 2.139351
      Morocco 1.984952
  South Korea 1.907058
   Uzbekistan 1.890281
United States 1.870043
        Spain 1.849919
    Argentina 1.811885
         Iran 1.747515
       Canada 1.715385
       Jordan 1.689882
xi=0.002  converged=False
         team  overall
        Japan 2.341983
    Australia 2.303121
       Mexico 2.277728
      Morocco 2.138581
        Spain 2.071749
  South Korea 1.976897
      Senegal 1.823817
       Jordan 1.790642
   Uzbekistan 1.753962
    Argentina 1.692009
       France 1.676223
United States 1.653633
xi=0.003  converged=False
       team  overall
  Australia 2.469026
     Mexico 2.406619
      Japan 2.327736
    Morocco 2.324455
      Spain 2.252389
South Korea 2.146090
    Senegal 2.081147
     Jordan 1.881272
     France 1.778372
 Uzbekistan 1.735258
   DR Congo 1.694960
     Norway 1.6830

In [7]:
%%time
# -- Fit Dixon-Coles -- EXP-16b: bounds anchor instead of equality constraint --
# L-BFGS-B handles bounds natively but NOT equality constraints.
# Fix: set bounds[eng_idx] = (0.0, 0.0) to anchor England attack=0.

init_params = np.zeros(2 * n_teams + 2)
init_params[2 * n_teams]     = 0.3
init_params[2 * n_teams + 1] = -0.1

eng_idx = team_idx.get('England', 0)
bounds  = [(None, None)] * (2 * n_teams + 2)
bounds[eng_idx] = (0.0, 0.0)  # anchor England attack = 0 (identifiability)

print('Fitting Dixon-Coles model (EXP-16b + EXP-17)...')
result = minimize(
    dc_log_likelihood_fast,
    init_params,
    method='L-BFGS-B',
    bounds=bounds,
    options={'maxiter': 2000, 'ftol': 1e-6}
)

attack_params  = result.x[:n_teams]
defense_params = result.x[n_teams:2*n_teams]
home_adv       = result.x[2*n_teams]
rho            = result.x[2*n_teams + 1]

print(f'Converged: {result.success}')
print(f'Home advantage factor: {np.exp(home_adv):.3f}x')
print(f'Rho (low-score correction): {rho:.4f}')
print(f'England attack anchor check: {attack_params[eng_idx]:.6f}  (should be ~0.0)')


Fitting Dixon-Coles model (EXP-16b + EXP-17)...
Converged: False
Home advantage factor: 1.331x
Rho (low-score correction): -0.0844
England attack anchor check: 0.000000  (should be ~0.0)
CPU times: total: 1min 25s
Wall time: 1min 28s


## EXP-26 — Refit DC on Top-Tier Matches Only

Qualifier/regional matches inflate weak-confederation teams (Japan #1, Australia #2).
Fix: fit DC only on World Cup + major continental tournaments.
6,353 matches across 201 teams — cleaner signal, no qualifier noise.
Teams not in top-tier get their ratings from the full fit as fallback.

In [8]:
%%time
# EXP-26: Refit DC on top-tier matches only (WC + major continentals)
# NOTE: params are NOT overwritten -- all-competitive fit stays active for ensemble
# This cell is for leaderboard analysis only
TOP_TIER = [
    "FIFA World Cup", "World Cup",
    "Copa America", "Copa América",
    "African Cup of Nations",
    "UEFA Euro", "European Championship", "UEFA Nations League",
    "AFC Asian Cup", "Asian Cup",
    "Gold Cup", "CONCACAF Championship", "CONCACAF Nations League",
    "Confederations Cup",
]
comp_top = matches[
    matches["tournament"].isin(TOP_TIER) &
    matches["home_score"].notna() &
    matches["away_score"].notna()
].copy()
comp_top["home_score"] = comp_top["home_score"].astype(int)
comp_top["away_score"] = comp_top["away_score"].astype(int)
comp_top["days_ago"]   = (comp_top["date"].max() - comp_top["date"]).dt.days
comp_top["weight"]     = np.exp(-XI * comp_top["days_ago"])

all_teams_top = sorted(set(comp_top["home_team"]) | set(comp_top["away_team"]))
team_idx_top  = {t: i for i, t in enumerate(all_teams_top)}
n_teams_top   = len(all_teams_top)

ht_top = np.array([team_idx_top[t] for t in comp_top["home_team"].values])
at_top = np.array([team_idx_top[t] for t in comp_top["away_team"].values])
hg_top = comp_top["home_score"].values
ag_top = comp_top["away_score"].values
w_top  = comp_top["weight"].values
m00t = (hg_top==0)&(ag_top==0); m01t=(hg_top==0)&(ag_top==1)
m10t = (hg_top==1)&(ag_top==0); m11t=(hg_top==1)&(ag_top==1)

def dc_nll_top(params):
    atk=params[:n_teams_top]; dfn=params[n_teams_top:2*n_teams_top]
    ha=params[2*n_teams_top]; rh=params[2*n_teams_top+1]
    lam=np.exp(atk[ht_top]+dfn[at_top]+ha); mu=np.exp(atk[at_top]+dfn[ht_top])
    tau=np.ones(len(hg_top))
    tau[m00t]=1-lam[m00t]*mu[m00t]*rh; tau[m01t]=1+lam[m01t]*rh
    tau[m10t]=1+mu[m10t]*rh;           tau[m11t]=1-rh
    valid=tau>0
    return -np.sum(w_top[valid]*(np.log(tau[valid])+poisson.logpmf(hg_top[valid],lam[valid])+poisson.logpmf(ag_top[valid],mu[valid])))

eng_i=team_idx_top.get("England",0)
init_t=np.zeros(2*n_teams_top+2); init_t[2*n_teams_top]=0.3; init_t[2*n_teams_top+1]=-0.1
bnds_t=[(None,None)]*(2*n_teams_top+2); bnds_t[eng_i]=(0.0,0.0)
res_top=minimize(dc_nll_top,init_t,method="L-BFGS-B",bounds=bnds_t,options={"maxiter":2000,"ftol":1e-6})

atk_top=res_top.x[:n_teams_top]; dfn_top=res_top.x[n_teams_top:2*n_teams_top]
print(f"EXP-26 converged: {res_top.success}, matches: {len(comp_top):,}, teams: {n_teams_top}")
df_top=pd.DataFrame({"team":all_teams_top,"attack":atk_top,"defense":dfn_top})
df_top["overall"]=df_top["attack"]-df_top["defense"]
df_top=df_top.sort_values("overall",ascending=False)
print("EXP-26 Top 20 (top-tier only, for reference):")
print(df_top.head(20)[["team","overall"]].to_string(index=False))
print("Global params unchanged -- all-competitive fit still active for ensemble.")


EXP-26 converged: False, matches: 6,318, teams: 190
EXP-26 Top 20 (top-tier only, for reference):
         team  overall
     Colombia 3.027956
       Brazil 2.658255
    Argentina 2.463595
    Venezuela 2.297859
      Uruguay 2.258966
      Morocco 2.182232
       Mexico 1.913500
        Spain 1.782791
     Portugal 1.758691
      Nigeria 1.621747
       France 1.600798
      Ecuador 1.546790
     Paraguay 1.492864
   Uzbekistan 1.451218
      Germany 1.378896
       Canada 1.310869
  Netherlands 1.262040
        Italy 1.252254
United States 1.195248
      Denmark 1.168910
Global params unchanged -- all-competitive fit still active for ensemble.
CPU times: total: 13.8 s
Wall time: 14.3 s


## Step 2: Attack/Defense Leaderboard

Inspect the fitted ratings — sanity check before using them as features.

In [9]:
dc_ratings = pd.DataFrame({
    'team': all_teams,
    'attack':  attack_params,
    'defense': defense_params,
})
# Lower defense = harder to score against (better defense)
dc_ratings['overall'] = dc_ratings['attack'] - dc_ratings['defense']
dc_ratings = dc_ratings.sort_values('overall', ascending=False)

print('Top 15 teams (overall = attack - defense):')
print(dc_ratings.head(15).to_string(index=False))
print()
print('Bottom 10 teams:')
print(dc_ratings.tail(10).to_string(index=False))

Top 15 teams (overall = attack - defense):
         team   attack   defense  overall
        Japan 1.213660 -1.128323 2.341983
    Australia 1.063305 -1.239816 2.303121
       Mexico 0.850938 -1.426790 2.277728
      Morocco 0.879996 -1.258585 2.138581
        Spain 1.166902 -0.904847 2.071749
  South Korea 1.008245 -0.968651 1.976897
      Senegal 0.772064 -1.051753 1.823817
       Jordan 1.008590 -0.782052 1.790642
   Uzbekistan 0.457427 -1.296535 1.753962
    Argentina 0.697756 -0.994253 1.692009
       France 0.732275 -0.943948 1.676223
United States 0.927290 -0.726344 1.653633
       Canada 0.806556 -0.735452 1.542008
      Germany 0.879003 -0.608944 1.487947
     Portugal 0.870708 -0.616843 1.487551

Bottom 10 teams:
          team    attack  defense   overall
      Anguilla -1.005204 1.153981 -2.159185
    Seychelles -1.222746 0.988488 -2.211233
        Brunei -1.209287 1.036636 -2.245922
         Macao -0.754513 1.524835 -2.279348
 Liechtenstein -1.457725 0.834539 -2.292264
   

## Step 3: Generate DC Probabilities for Every Match

For each match in train/test, compute P(home win), P(draw), P(away win) using fitted parameters **as of that match date** — we refit per-match using only prior data to avoid leakage.

**Practical approach:** Use a single global fit (already done) but only use matches *before* each target match for computing team weights. This is the standard implementation used in practice.

In [10]:
MAX_GOALS = 10
_goals    = np.arange(MAX_GOALS + 1)

# Outcome masks — built once, reused for every match
_home_mask = _goals[:, None] > _goals[None, :]   # i > j → home win
_draw_mask = _goals[:, None] == _goals[None, :]  # i == j → draw
_away_mask = _goals[:, None] < _goals[None, :]   # i < j → away win

def dc_match_probs(home_team, away_team, attack_params, defense_params,
                   home_adv, rho, team_idx, neutral=False):
    """Vectorized Dixon-Coles: full 11x11 score matrix via numpy broadcasting."""
    if home_team not in team_idx or away_team not in team_idx:
        return 0.45, 0.25, 0.30

    hi  = team_idx[home_team]
    ai  = team_idx[away_team]
    ha  = home_adv if not neutral else 0.0
    lam = np.exp(attack_params[hi] + defense_params[ai] + ha)
    mu  = np.exp(attack_params[ai] + defense_params[hi])

    # Poisson PMF vectors then outer product → full score probability matrix
    score_matrix = np.outer(poisson.pmf(_goals, lam), poisson.pmf(_goals, mu))

    # Apply tau correction to the 4 low-score cells only
    score_matrix[0, 0] *= max(1 - lam * mu * rho, 1e-10)
    score_matrix[0, 1] *= max(1 + lam * rho,       1e-10)
    score_matrix[1, 0] *= max(1 + mu  * rho,       1e-10)
    score_matrix[1, 1] *= max(1 - rho,             1e-10)

    hw = np.sum(score_matrix * _home_mask)
    dr = np.sum(score_matrix * _draw_mask)
    aw = np.sum(score_matrix * _away_mask)
    total = hw + dr + aw
    return hw/total, dr/total, aw/total

print('Vectorized DC function defined.')

# Sanity checks
for ht, at in [('Spain','Qatar'), ('Argentina','France'), ('Morocco','Saudi Arabia')]:
    hw, dr, aw = dc_match_probs(ht, at, attack_params, defense_params,
                                 home_adv, rho, team_idx, neutral=True)
    print(f'{ht} vs {at}: home={hw:.3f}, draw={dr:.3f}, away={aw:.3f}')

Vectorized DC function defined.
Spain vs Qatar: home=0.782, draw=0.147, away=0.072
Argentina vs France: home=0.313, draw=0.382, away=0.306
Morocco vs Saudi Arabia: home=0.524, draw=0.337, away=0.139


In [11]:
%%time
train_df = pd.read_csv(PROCESSED_DIR / 'train.csv')
test_df  = pd.read_csv(PROCESSED_DIR / 'test.csv')

def add_dc_features_fast(df):
    """Vectorized DC features including lambda, mu, and derived dominance features."""
    ht = df['home_team'].values
    at = df['away_team'].values
    neutral_arr = df['neutral.1'].values.astype(bool)

    hi_arr = np.array([team_idx.get(t, -1) for t in ht])
    ai_arr = np.array([team_idx.get(t, -1) for t in at])
    known  = (hi_arr >= 0) & (ai_arr >= 0)
    ha_arr = np.where(neutral_arr, 0.0, home_adv)

    lam = np.where(known,
                   np.exp(attack_params[np.where(known, hi_arr, 0)]
                          + defense_params[np.where(known, ai_arr, 0)]
                          + ha_arr), 1.3)
    mu  = np.where(known,
                   np.exp(attack_params[np.where(known, ai_arr, 0)]
                          + defense_params[np.where(known, hi_arr, 0)]), 1.1)

    # Score matrix (n, 11, 11)
    home_pmf  = poisson.pmf(_goals[None, :], lam[:, None])
    away_pmf  = poisson.pmf(_goals[None, :], mu[:, None])
    score_mat = home_pmf[:, :, None] * away_pmf[:, None, :]

    score_mat[:, 0, 0] *= np.maximum(1 - lam * mu * rho, 1e-10)
    score_mat[:, 0, 1] *= np.maximum(1 + lam * rho,       1e-10)
    score_mat[:, 1, 0] *= np.maximum(1 + mu  * rho,       1e-10)
    score_mat[:, 1, 1] *= np.maximum(1 - rho,             1e-10)

    hw    = np.sum(score_mat * _home_mask[None, :, :], axis=(1, 2))
    dr    = np.sum(score_mat * _draw_mask[None, :, :], axis=(1, 2))
    aw    = np.sum(score_mat * _away_mask[None, :, :], axis=(1, 2))
    total = hw + dr + aw

    out = df.copy()
    # EXP-15 features (win/draw/loss probs)
    out['dc_home_win_prob']   = hw / total
    out['dc_draw_prob']       = dr / total
    out['dc_away_win_prob']   = aw / total
    # EXP-18: raw expected goals (dominance signal)
    out['dc_lambda']          = lam   # expected home goals
    out['dc_mu']              = mu    # expected away goals
    # EXP-19: derived dominance features
    out['dc_total_goals']     = lam + mu   # low = draw likely, high = open game
    out['dc_goal_diff']       = lam - mu   # positive = home favored, magnitude = dominance
    return out

print('Adding DC features to train set...')
train_dc = add_dc_features_fast(train_df)
print('Adding DC features to test set...')
test_dc  = add_dc_features_fast(test_df)

print('Done. New DC features: dc_home_win_prob, dc_draw_prob, dc_away_win_prob,')
print('                       dc_lambda, dc_mu, dc_total_goals, dc_goal_diff')
print(train_dc[['home_team','away_team','dc_lambda','dc_mu','dc_total_goals','dc_goal_diff']].head(5))


Adding DC features to train set...
Adding DC features to test set...
Done. New DC features: dc_home_win_prob, dc_draw_prob, dc_away_win_prob,
                       dc_lambda, dc_mu, dc_total_goals, dc_goal_diff
          home_team         away_team  dc_lambda     dc_mu  dc_total_goals  \
0  Northern Ireland          Scotland   0.962963  1.252332        2.215294   
1             Wales  Northern Ireland   1.462918  0.984158        2.447076   
2  Northern Ireland           England   0.391320  0.766333        1.157653   
3          Scotland           England   0.580867  0.656933        1.237800   
4             Wales           England   0.509620  0.893938        1.403558   

   dc_goal_diff  
0     -0.289369  
1      0.478760  
2     -0.375013  
3     -0.076066  
4     -0.384318  
CPU times: total: 406 ms
Wall time: 535 ms


## Step 4: Train Ensemble with DC Features

In [12]:
BASE_FEATURES = [
    'home_elo_before', 'away_elo_before', 'elo_diff',
    'home_win_rate_5', 'home_avg_scored_5', 'home_avg_conceded_5',
    'home_pts_per_match_5', 'home_matches_played_5',
    'home_win_rate_10', 'home_avg_scored_10', 'home_avg_conceded_10',
    'home_pts_per_match_10', 'home_matches_played_10',
    'away_win_rate_5', 'away_avg_scored_5', 'away_avg_conceded_5',
    'away_pts_per_match_5', 'away_matches_played_5',
    'away_win_rate_10', 'away_avg_scored_10', 'away_avg_conceded_10',
    'away_pts_per_match_10', 'away_matches_played_10',
    'h2h_home_win_rate', 'h2h_home_avg_scored', 'h2h_home_avg_conceded',
    'h2h_total_meetings', 'h2h_recent_win_rate',
    'neutral.1', 'tournament_importance',
    'home_conf_UEFA', 'home_conf_CAF', 'home_conf_AFC',
    'home_conf_CONCACAF', 'home_conf_CONMEBOL', 'home_conf_OFC', 'home_conf_UNKNOWN',
    'away_conf_UEFA', 'away_conf_CAF', 'away_conf_AFC',
    'away_conf_CONCACAF', 'away_conf_CONMEBOL', 'away_conf_OFC', 'away_conf_UNKNOWN',
    'same_confederation',
]

def add_engineered_features(df):
    d = df.copy()
    d['elo_diff_sq']         = d['elo_diff'] ** 2 * np.sign(d['elo_diff'])
    d['home_form_momentum']  = d['home_win_rate_5'] - d['home_win_rate_10']
    d['away_form_momentum']  = d['away_win_rate_5'] - d['away_win_rate_10']
    d['home_goal_diff_form'] = d['home_avg_scored_5'] - d['home_avg_conceded_5']
    d['away_goal_diff_form'] = d['away_avg_scored_5'] - d['away_avg_conceded_5']
    d['net_goal_diff']       = d['home_goal_diff_form'] - d['away_goal_diff_form']
    d['h2h_confidence']      = d['h2h_recent_win_rate'] * (d['h2h_total_meetings'] / (d['h2h_total_meetings'] + 5))
    return d

train_eng = add_engineered_features(train_dc)
test_eng  = add_engineered_features(test_dc)

FEATURE_COLS = BASE_FEATURES + [
    'elo_diff_sq', 'home_form_momentum', 'away_form_momentum',
    'home_goal_diff_form', 'away_goal_diff_form', 'net_goal_diff', 'h2h_confidence',
    'dc_home_win_prob', 'dc_draw_prob', 'dc_away_win_prob',  # EXP-15
    'dc_lambda', 'dc_mu', 'dc_total_goals', 'dc_goal_diff',          # EXP-18/19
]

X_train = train_eng[FEATURE_COLS].values
X_test  = test_eng[FEATURE_COLS].values

le = LabelEncoder()
y_train = le.fit_transform(train_dc['outcome'].values)
y_test  = le.transform(test_dc['outcome'].values)

classes = np.unique(y_train)
weights = compute_class_weight('balanced', classes=classes, y=y_train)
cw      = dict(zip(classes, weights))

from sklearn.preprocessing import StandardScaler
scaler  = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f'Feature set: {len(FEATURE_COLS)} features (52 original + 7 DC)')
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

Feature set: 59 features (52 original + 7 DC)
Train: (35304, 59) | Test: (3313, 59)


In [13]:
%%time
# Same best ensemble from AutoResearch + 3 DC features
xgb_base = xgb.XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0,
    eval_metric='mlogloss', random_state=42, verbosity=0,
)
xgb_model = CalibratedClassifierCV(xgb_base, method='isotonic', cv=5)

rf_base = RandomForestClassifier(
    n_estimators=300, max_depth=12, min_samples_leaf=5,
    class_weight=cw, random_state=42, n_jobs=-1,
)
rf_model = CalibratedClassifierCV(rf_base, method='isotonic', cv=5)

lr_model = LogisticRegression(
    C=1.0, max_iter=1000, class_weight=cw,
    solver='lbfgs', multi_class='multinomial', random_state=42,
)

ensemble = VotingClassifier(
    estimators=[('xgb', xgb_model), ('rf', rf_model), ('lr', lr_model)],
    voting='soft', weights=[4, 2, 2], n_jobs=-1,
)

print('Training ensemble with DC features...')
ensemble.fit(X_train_s, y_train)
print('Done.')

Training ensemble with DC features...
Done.
CPU times: total: 1.75 s
Wall time: 1min


## Step 5: Evaluate

In [14]:
y_pred       = ensemble.predict(X_test_s)
y_pred_proba = ensemble.predict_proba(X_test_s)

acc = accuracy_score(y_test, y_pred)
f1  = f1_score(y_test, y_pred, average='macro')
ll  = log_loss(y_test, y_pred_proba)

BASELINE_LL  = 0.8263
BASELINE_ACC = 0.6242

print('=' * 55)
print('  Ensemble + Dixon-Coles Features')
print('=' * 55)
print(f'  Accuracy : {acc:.4f}  (baseline: {BASELINE_ACC})')
print(f'  F1 Macro : {f1:.4f}')
print(f'  Log Loss : {ll:.4f}  (baseline: {BASELINE_LL})')
delta = BASELINE_LL - ll
print(f'  Delta LL : {delta:+.4f}  ({"IMPROVEMENT" if delta > 0 else "WORSE"})')
print()

from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred, target_names=le.classes_))

  Ensemble + Dixon-Coles Features
  Accuracy : 0.6266  (baseline: 0.6242)
  F1 Macro : 0.5045
  Log Loss : 0.8131  (baseline: 0.8263)
  Delta LL : +0.0132  (IMPROVEMENT)

              precision    recall  f1-score   support

    away_win       0.58      0.63      0.60       894
        draw       0.41      0.10      0.16       750
    home_win       0.67      0.86      0.75      1669

    accuracy                           0.63      3313
   macro avg       0.55      0.53      0.50      3313
weighted avg       0.58      0.63      0.58      3313



In [15]:
# Full comparison table
comparison = pd.DataFrame([
    {'Model': 'Notebook 04 Ensemble (XGB+RF+LR)',       'Log Loss': 0.8333, 'Accuracy': 0.6212, 'F1': 0.5313},
    {'Model': 'AutoResearch best (calibrated+eng)',      'Log Loss': 0.8263, 'Accuracy': 0.6242, 'F1': 0.4988},
    {'Model': 'TabNet',                                  'Log Loss': 0.8600, 'Accuracy': 0.6004, 'F1': 0.5325},
    {'Model': 'EXP-15: Ensemble + DC probs (3 feat)',    'Log Loss': 0.8148, 'Accuracy': 0.6278, 'F1': 0.5046},
    {'Model': 'EXP-16/18/19: DC probs+lambda+mu (7 feat)', 'Log Loss': round(ll,4), 'Accuracy': round(acc,4), 'F1': round(f1,4)},
])
print(comparison.to_string(index=False))

# Save comparison
comparison.to_csv(PROCESSED_DIR / 'model_comparison_final.csv', index=False)

                                    Model  Log Loss  Accuracy     F1
         Notebook 04 Ensemble (XGB+RF+LR)    0.8333    0.6212 0.5313
       AutoResearch best (calibrated+eng)    0.8263    0.6242 0.4988
                                   TabNet    0.8600    0.6004 0.5325
     EXP-15: Ensemble + DC probs (3 feat)    0.8148    0.6278 0.5046
EXP-16/18/19: DC probs+lambda+mu (7 feat)    0.8131    0.6266 0.5045


In [16]:
# Save DC-augmented train/test CSVs for AutoResearch (EXP-24)
train_eng_dc = add_engineered_features(train_dc)
test_eng_dc  = add_engineered_features(test_dc)

train_eng_dc.to_csv(PROCESSED_DIR / "train_dc.csv", index=False)
test_eng_dc.to_csv(PROCESSED_DIR  / "test_dc.csv",  index=False)

print(f"Saved train_dc.csv: {len(train_eng_dc)} rows, {len(train_eng_dc.columns)} cols")
print(f"Saved test_dc.csv:  {len(test_eng_dc)} rows, {len(test_eng_dc.columns)} cols")


Saved train_dc.csv: 35304 rows, 71 cols
Saved test_dc.csv:  3313 rows, 71 cols


In [17]:
# ── Save DC ratings ────────────────────────────────────────────────────────────
dc_ratings_df = pd.DataFrame({'team': all_teams, 'attack': attack_params, 'defense': defense_params})
dc_ratings_df['overall'] = dc_ratings_df['attack'] - dc_ratings_df['defense']
dc_ratings_df.to_csv(PROCESSED_DIR / 'dc_ratings.csv', index=False)

# Save model params for tournament simulation
dc_params = {
    'home_adv': float(home_adv),
    'rho': float(rho),
    'xi': XI,
    'reference_date': str(reference_date.date()),
    'n_teams': n_teams,
}
with open(MODELS_DIR / 'dc_params.json', 'w') as f:
    json.dump(dc_params, f, indent=2)

# Save best model
joblib.dump(ensemble, MODELS_DIR / 'best_model_dc.pkl')
joblib.dump(scaler,   MODELS_DIR / 'scaler_dc.pkl')
joblib.dump(le,       MODELS_DIR / 'label_encoder_dc.pkl')
joblib.dump(FEATURE_COLS, MODELS_DIR / 'feature_cols_dc.pkl')
joblib.dump({'attack': attack_params, 'defense': defense_params,
             'teams': all_teams, 'team_idx': team_idx},
            MODELS_DIR / 'dc_model.pkl')

print('Saved:')
print('  data/processed/dc_ratings.csv')
print('  models/dc_params.json')
print('  models/best_model_dc.pkl  ← use this for tournament simulation')
print('  models/dc_model.pkl       ← attack/defense ratings for all teams')
print(f'\nFinal best log_loss: {ll:.4f}')

Saved:
  data/processed/dc_ratings.csv
  models/dc_params.json
  models/best_model_dc.pkl  ← use this for tournament simulation
  models/dc_model.pkl       ← attack/defense ratings for all teams

Final best log_loss: 0.8131
